# Práctica RAG - Fase 2: Benchmark de Embeddings

Este notebook ejecuta la **Fase 2** de la práctica: comparar 3 modelos de embedding locales sobre dos corpus distintos para determinar cuál ofrece la mejor capacidad de recuperación (*retrieval*) antes de pasar a la generación con LLM.

###  Los dos documentos evaluados:
1. **Normativa jurídica:** *Estatuto de los Trabajadores* (BOE consolidado, 420 fragmentos).
2. **Documentación técnica:** *Manual del Administrador de Debian* (1.210 fragmentos con comandos y rutas).

###  Modelos candidatos:
- **`EmbeddingGemma-2`** (`google/embeddinggemma-2`): 0.7B parámetros, arquitectura moderna de Google DeepMind.
- **`Spanish-BETO`** (`hiiamsid/sentence_similarity_spanish_es`): 0.11B parámetros, especialista exclusivo en español.
- **`Multilingual-E5-Small`** (`intfloat/multilingual-e5-small`): 0.12B parámetros, modelo multilingüe de Microsoft con prefijos (`query:` / `passage:`).

In [1]:
import json
import time
import os
import torch
from sentence_transformers import SentenceTransformer, util

# Comprobar aceleración por GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo de cómputo en uso: {device}')
if device == 'cuda':
    print(f'GPU detectada: {torch.cuda.get_device_name(0)}')
    print(f'VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

/home/g4/reto_1_2-2627-Rag-Benchmark/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo de cómputo en uso: cuda
GPU detectada: NVIDIA A800 40GB Active MIG 3g.20gb
VRAM disponible: 19.62 GB


## 1. Definición de Candidatos y Documentos

In [2]:
MODELOS = [
    {
        'nombre': 'google/embeddinggemma-2',
        'alias': 'EmbeddingGemma-2',
        'params': '0.7B',
        'prefijo_query': '',
        'prefijo_doc': '',
        'trust_remote_code': True
    },
    {
        'nombre': 'hiiamsid/sentence_similarity_spanish_es',
        'alias': 'Spanish-BETO',
        'params': '0.11B',
        'prefijo_query': '',
        'prefijo_doc': '',
        'trust_remote_code': False
    },
    {
        'nombre': 'intfloat/multilingual-e5-small',
        'alias': 'Multilingual-E5-Small',
        'params': '0.12B',
        'prefijo_query': 'query: ',
        'prefijo_doc': 'passage: ',
        'trust_remote_code': False
    }
]

DOCUMENTOS = [
    {
        'nombre': '1. Normativa (BOE Estatuto de los Trabajadores)',
        'archivo_chunks': 'data/processed/boe_chunks.json',
        'archivo_preguntas': 'preguntas/boe_questions.json'
    },
    {
        'nombre': '2. Técnica (Debian Handbook)',
        'archivo_chunks': 'data/processed/debian_chunks.json',
        'archivo_preguntas': 'preguntas/debian_questions.json'
    }
]

## 2. Funciones de Evaluación (Recall@k y MRR)

Siguiendo los criterios de evaluación del proyecto:
- **Recall@k:** Porcentaje de preguntas donde el fragmento correcto (*ground truth*) aparece en los primeros $k$ resultados ($k=1, 3, 5$).
- **MRR (Mean Reciprocal Rank):** Mide la posición del primer resultado correcto ($1 / \text{rango}$). Premia a los modelos que colocan el fragmento acertado en el puesto 1.

In [3]:
def calcular_metricas(ranking_resultados, preguntas_con_respuesta):
    total = len(preguntas_con_respuesta)
    if total == 0:
        return 0, 0, 0, 0

    hits_at_1 = 0
    hits_at_3 = 0
    hits_at_5 = 0
    reciprocal_ranks = []

    for q, recuperados in ranking_resultados:
        esperados = set(q['chunks_esperados'])
        if not esperados:
            continue  # Las preguntas sin respuesta se excluyen del cálculo de Recall

        top_1 = set(recuperados[:1])
        top_3 = set(recuperados[:3])
        top_5 = set(recuperados[:5])

        if top_1 & esperados:
            hits_at_1 += 1
        if top_3 & esperados:
            hits_at_3 += 1
        if top_5 & esperados:
            hits_at_5 += 1

        primer_rango = None
        for i, chunk_id in enumerate(recuperados, start=1):
            if chunk_id in esperados:
                primer_rango = i
                break

        if primer_rango:
            reciprocal_ranks.append(1.0 / primer_rango)
        else:
            reciprocal_ranks.append(0.0)

    recall_1 = hits_at_1 / total
    recall_3 = hits_at_3 / total
    recall_5 = hits_at_5 / total
    mrr = sum(reciprocal_ranks) / total if reciprocal_ranks else 0.0

    return recall_1, recall_3, recall_5, mrr

## 3. Función de Evaluación por Documento y Medición de GPU

In [4]:
def evaluar_modelo_en_documento(modelo_info, documento):
    with open(documento['archivo_chunks'], 'r', encoding='utf-8') as f:
        chunks = json.load(f)
    with open(documento['archivo_preguntas'], 'r', encoding='utf-8') as f:
        preguntas = json.load(f)

    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    if dev == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

    # 1. Cargar modelo y obtener dimensión
    modelo = SentenceTransformer(
        modelo_info['nombre'],
        device=dev,
        trust_remote_code=modelo_info['trust_remote_code']
    )
    dimension = modelo.get_embedding_dimension()

    # 2. Medir tiempo de indexado
    textos_docs = [modelo_info['prefijo_doc'] + c['text'] for c in chunks]
    start_time = time.time()
    embeddings_docs = modelo.encode(
        textos_docs,
        batch_size=32,
        show_progress_bar=False,
        convert_to_tensor=True,
        normalize_embeddings=True
    )
    tiempo_indexado = time.time() - start_time
    tamano_mb = (len(chunks) * dimension * 4) / (1024 * 1024)

    # 3. Vectorizar preguntas y recuperar top-5
    preguntas_con_respuesta = [q for q in preguntas if q['chunks_esperados']]
    ranking_resultados = []

    for q in preguntas:
        query_text = modelo_info['prefijo_query'] + q['pregunta']
        query_emb = modelo.encode(query_text, convert_to_tensor=True, normalize_embeddings=True)
        cos_scores = util.cos_sim(query_emb, embeddings_docs)[0]
        top_indices = torch.topk(cos_scores, k=min(10, len(chunks))).indices.tolist()
        recuperados = [chunks[idx]['id'] for idx in top_indices]
        ranking_resultados.append((q, recuperados))

    # 4. Métricas de búsqueda
    r1, r3, r5, mrr = calcular_metricas(ranking_resultados, preguntas_con_respuesta)

    # 5. Medir VRAM pico
    vram_gb = 0.0
    if dev == 'cuda':
        vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    del modelo
    del embeddings_docs
    if dev == 'cuda':
        torch.cuda.empty_cache()

    return {
        'modelo': modelo_info['alias'],
        'nombre_hf': modelo_info['nombre'],
        'parametros': modelo_info['params'],
        'dimension': dimension,
        'recall_1': round(r1, 3),
        'recall_3': round(r3, 3),
        'recall_5': round(r5, 3),
        'mrr': round(mrr, 3),
        'tiempo_indexado_s': round(tiempo_indexado, 2),
        'tamano_mb': round(tamano_mb, 2),
        'vram_gb': round(vram_gb, 2)
    }

## 4. Ejecución del Benchmark y Generación de Tablas A

In [5]:
os.makedirs('resultados', exist_ok=True)
resultados_totales = {}

print('=' * 70)
print('INICIANDO BENCHMARK DE EMBEDDINGS (FASE 2)')
print('=' * 70)

for esc in DOCUMENTOS:
    print(f"\n>>> Evaluando Documento: {esc['nombre']}")
    resultados_totales[esc['nombre']] = []

    for m_info in MODELOS:
        print(f"  -> Probando modelo: {m_info['alias']}...")
        res = evaluar_modelo_en_documento(m_info, esc)
        resultados_totales[esc['nombre']].append(res)
        print(f"     Recall@5: {res['recall_5']} | MRR: {res['mrr']} | Indexado: {res['tiempo_indexado_s']}s | VRAM: {res['vram_gb']} GB")

with open('resultados/embeddings_benchmark.json', 'w', encoding='utf-8') as f:
    json.dump(resultados_totales, f, ensure_ascii=False, indent=2)
print('\nResultados guardados correctamente en resultados/embeddings_benchmark.json')

INICIANDO BENCHMARK DE EMBEDDINGS (FASE 2)

>>> Evaluando Documento: 1. Normativa (BOE Estatuto de los Trabajadores)
  -> Probando modelo: EmbeddingGemma-2...


Loading weights:   0%|                                | 0/1376 [00:00<?, ?it/s]

Loading weights:  51%|██████████▏         | 700/1376 [00:00<00:00, 6915.12it/s]

Loading weights: 100%|███████████████████| 1376/1376 [00:00<00:00, 6148.73it/s]

     Recall@5: 1.0 | MRR: 0.917 | Indexado: 2.14s | VRAM: 2.82 GB
  -> Probando modelo: Spanish-BETO...


Loading weights:   0%|                                 | 0/199 [00:00<?, ?it/s]

... (11 outputs left out)

     Recall@5: 0.167 | MRR: 0.049 | Indexado: 14.01s | VRAM: 2.37 GB
  -> Probando modelo: Multilingual-E5-Small...


Loading weights:   0%|                                 | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 5950.81it/s]

     Recall@5: 0.5 | MRR: 0.278 | Indexado: 3.99s | VRAM: 1.8 GB

Resultados guardados correctamente en resultados/embeddings_benchmark.json


## 5. Visualización Formateada de las Tablas A

In [6]:
import pandas as pd
from IPython.display import display, Markdown

for nombre_doc, filas in resultados_totales.items():
    display(Markdown(f"###  {nombre_doc}"))
    
    df = pd.DataFrame(filas)
    df = df[[
        'modelo', 'parametros', 'dimension', 
        'recall_1', 'recall_3', 'recall_5', 'mrr', 
        'tiempo_indexado_s', 'tamano_mb', 'vram_gb'
    ]]
    df.columns = [
        'Modelo', 'Parámetros', 'Dimensión', 
        'Recall@1', 'Recall@3', 'Recall@5', 'MRR', 
        'Indexado (s)', 'Índice (MB)', 'VRAM (GB)'
    ]
    
    # Formatear números para lectura clara
    formato = {
        'Recall@1': '{:.3f}',
        'Recall@3': '{:.3f}',
        'Recall@5': '{:.3f}',
        'MRR': '{:.3f}',
        'Indexado (s)': '{:.2f} s',
        'Índice (MB)': '{:.2f} MB',
        'VRAM (GB)': '{:.2f} GB'
    }
    tabla_estilizada = df.style.format(formato).hide(axis='index')
    display(tabla_estilizada)


###  1. Normativa (BOE Estatuto de los Trabajadores)

Modelo,Parámetros,Dimensión,Recall@1,Recall@3,Recall@5,MRR,Indexado (s),Índice (MB),VRAM (GB)
EmbeddingGemma-2,0.7B,768,0.833,1.000,1.000,0.917,2.14 s,1.23 MB,2.82 GB
Spanish-BETO,0.11B,768,0.583,0.833,0.833,0.722,2.33 s,1.23 MB,2.13 GB
Multilingual-E5-Small,0.12B,384,0.750,0.833,0.917,0.812,0.94 s,0.62 MB,1.81 GB


###  2. Técnica (Debian Handbook)

Modelo,Parámetros,Dimensión,Recall@1,Recall@3,Recall@5,MRR,Indexado (s),Índice (MB),VRAM (GB)
EmbeddingGemma-2,0.7B,768,0.083,0.333,0.417,0.253,9.79 s,3.54 MB,9.97 GB
Spanish-BETO,0.11B,768,0.000,0.000,0.167,0.049,14.01 s,3.54 MB,2.37 GB
Multilingual-E5-Small,0.12B,384,0.167,0.333,0.500,0.278,3.99 s,1.77 MB,1.80 GB


## 6. Conclusiones y Selección de Ganadores

- **Para el Documento 1 (BOE):** Se elige **`EmbeddingGemma-2`** porque alcanza un **Recall@5 de 1.0 (100%)** y un **MRR de 0.917**, encontrando el artículo jurídico exacto en los primeros puestos en todas las consultas.
- **Para el Documento 2 (Debian):** Se elige **`Multilingual-E5-Small`** porque obtiene el **mayor Recall@5 (0.500)** y el mejor MRR (0.278), con un coste computacional ínfimo (3.97 s de indexado y 1.80 GB de VRAM).
- **Decisión global del grupo:** Como elección global para el sistema completo, nos quedaríamos con **`Multilingual-E5-Small`** de Microsoft. Es el más rápido con diferencia (indexa en menos de 1 segundo en el BOE y menos de 4 segundos en Debian), genera índices mínimos (0.62 MB y 1.77 MB) y gasta muy poca memoria VRAM (1.80 GB). Aunque en el documento del BOE no quedó primero (0.917 frente al 1.000 de EmbeddingGemma-2), en Debian fue el claro ganador y en conjunto es el modelo más consistente, barato y eficiente para producción.